# Limpieza y preparación — Cancelaciones hoteleras

## Alcance de la fase 3
Preparar una representación reproducible que separe **predictores, etiqueta y metadatos**. Se han revisado el EDA ejecutado, los informes de fases 1–2, README y los proyectos Python/Estadística. Se conserva su organización (`data/`, `notebooks/`, `src/`), funciones sencillas y explicación en español; no se trasladan sus reglas de eliminación porque no hay evidencia equivalente en estas reservas.

No se repite el EDA completo, crean particiones ni entrenan modelos predictivos. Se definen transformadores nuevos **sin ajustar sobre reservas reales**. Las pruebas de `fit` usan solo tres registros inventados. El único CSV preparado conserva todas las filas y los valores desconocidos pendientes de imputación.

## 1. Carga y revisión dirigida del EDA
**Pregunta:** ¿qué hallazgos requieren una decisión de preparación?

In [1]:
# EDICIÓN COMENTADA PARA LECTURA. Las fuentes ejecutadas y selladas permanecen en la raíz.
# Cargar datos, reglas y dependencias y registrar huellas antes de preparar.
# Solo se añaden comentarios: no se han ejecutado instrucciones de esta copia.
from pathlib import Path
import sys
import hashlib
import json
import io
import unittest
import platform
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/raw/hotel_bookings.csv").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.cleaning import clean
from src.features import (FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES, LEAKAGE_FEATURES,
                          UNCERTAIN_FEATURES, build_features, separate_model_data, audit_availability)
from src.preprocessing import build_preprocessor
from src.preparation import run_preparation, TEMPORAL_PLAN

DATA_PATH = PROJECT_ROOT / "data/raw/hotel_bookings.csv"
REPORT_PATH = PROJECT_ROOT / "reports"
# Leer las columnas y filas solicitadas; los filtros de esta llamada determinan la población disponible.
raw = pd.read_csv(DATA_PATH, encoding="utf-8", encoding_errors="strict")
raw_before = raw.copy(deep=True)
with DATA_PATH.open("rb") as file:
    # Calcular una huella determinista para trazabilidad; no usarla como característica predictiva.
    original_sha = hashlib.file_digest(file, "sha256").hexdigest()
eda_summary = json.loads((REPORT_PATH / "eda_resumen.json").read_text(encoding="utf-8"))
# La huella debe conservarse: cualquier cambio de datos o evidencia detiene la comprobación.
assert original_sha == eda_summary["sha256_csv"]
# La huella debe conservarse: cualquier cambio de datos o evidencia detiene la comprobación.
assert raw.shape == (119390, 32)
display(pd.DataFrame(eda_summary["nulos"]))
display(pd.DataFrame(eda_summary["anomalias"]))
print("Duplicados adicionales según EDA:", eda_summary["duplicados_adicionales"])

,variable,nulos,porcentaje
0,company,112593,94.306893
1,agent,16340,13.686238
2,country,488,0.408744
3,children,4,0.003350


,comprobacion,registros,porcentaje
0,ADR negativo,1,0.000838
1,ADR cero (puede ser válido),1959,1.640841
2,Cero noches,715,0.598878
3,Cero huéspedes conocidos,180,0.150766
4,Huéspedes desconocidos,4,0.003350
5,Más de 10 huéspedes (revisar),14,0.011726
6,Conteos/antelación negativos,0,0.000000
7,Llegada inválida,0,0.000000
8,Fecha estado inválida,0,0.000000
9,Repetidor fuera de 0/1,0,0.000000


Duplicados adicionales según EDA: 31994


**Decisión.** El tratamiento se guía por semántica y disponibilidad temporal, no por mejorar una métrica. Se conservan duplicados, ausentes y casos sospechosos cuando no se conoce una corrección verificable. Las reglas siguientes no usan la etiqueta para decidir qué reservas mantener.

## 2. Duplicados: conservar con trazabilidad
**Pregunta:** ¿podemos demostrar que los registros repetidos representan la misma reserva?

In [2]:
# EDICIÓN COMENTADA PARA LECTURA. Las fuentes ejecutadas y selladas permanecen en la raíz.
# Comparar el efecto hipotético de deduplicar; la decisión aplicada conserva todas las filas.
# Solo se añaden comentarios: no se han ejecutado instrucciones de esta copia.
duplicated = raw.duplicated(keep=False)
duplicates = raw.duplicated()
impact = pd.DataFrame({
    "escenario": ["Conservar todos (decisión)", "Quitar repeticiones (solo cálculo hipotético)"],
    "registros": [len(raw), len(raw) - int(duplicates.sum())],
    "cancelaciones": [int(raw.is_canceled.sum()), int(raw.is_canceled.sum() - raw.loc[duplicates, "is_canceled"].sum())]
})
impact["tasa_pct"] = 100 * impact.cancelaciones / impact.registros
display(impact)
print("Registros en grupos repetidos, incluidas primeras apariciones:", int(duplicated.sum()))

,escenario,registros,cancelaciones,tasa_pct
0,Conservar todos (decisión),119390,44224,37.041628
1,Quitar repeticiones (solo cálculo hipotético),87396,24025,27.489816


Registros en grupos repetidos, incluidas primeras apariciones: 40165


**Interpretación y decisión.** No hay ID único de reserva. Filas idénticas podrían ser reservas distintas de grupos o canales con características iguales. El segundo escenario es una resta de recuentos para dimensionar el impacto: **no se construye ni exporta un dataset deduplicado**. Conservar mantiene la distribución observada, pero repeticiones técnicas podrían dar peso excesivo a algunos patrones y exagerar el tamaño efectivo de muestra. Eliminar sin evidencia podría borrar demanda real y alterar las tasas.

Se conserva el orden y se asigna `meta_row_id` como referencia técnica, no como ID real de reserva. `meta_duplicate_group` agrupa huellas de las 32 columnas originales (incluida etiqueta/estado), exclusivamente para auditoría; no es predictor. En fase 4 se verificará que duplicados no crucen conjuntos. Las huellas de 64 bits tienen un riesgo teórico de colisión; no sustituyen un identificador operativo.

## 3. Valores ausentes y corrección de tipos
**Pregunta:** ¿qué es desconocido y qué puede significar “no aplicable”?

In [3]:
# EDICIÓN COMENTADA PARA LECTURA. Las fuentes ejecutadas y selladas permanecen en la raíz.
# Aplicar tipos y explicar ausencias sin aprender medianas de todo el archivo.
# Solo se añaden comentarios: no se han ejecutado instrucciones de esta copia.
cleaned = clean(raw)
missing_policy = pd.DataFrame([
    ("children", int(raw.children.isna().sum()), "Desconocido", "Int64 nullable; mantener NA y children_missing. Mediana solo al ajustar con X_train."),
    ("country", int(raw.country.isna().sum()), "Origen desconocido/no informado", "Mantener NA; categoría Desconocido mediante imputador del futuro pipeline."),
    ("agent", int(raw.agent.isna().sum()), "No informado o no intervino agente; no distinguible", "ID como cadena sin .0 y NA conservado; fuera de X por incertidumbre temporal."),
    ("company", int(raw.company.isna().sum()), "No informado o no hay empresa asociada; no distinguible", "ID como cadena sin .0 y NA conservado; fuera de X por incertidumbre temporal.")
], columns=["variable", "nulos", "significado", "tratamiento"])
display(missing_policy)
display(pd.DataFrame({"tipo_original": raw[["children", "agent", "company"]].dtypes.astype(str), "tipo_limpio": cleaned[["children", "agent", "company"]].dtypes.astype(str)}))
display(cleaned.loc[raw.children.isna(), ["hotel", "children", "country", "agent", "company"]])

,variable,nulos,significado,tratamiento
0,children,4,Desconocido,Int64 nullable; mantener NA y children_missing...
1,country,488,Origen desconocido/no informado,Mantener NA; categoría Desconocido mediante im...
2,agent,16340,No informado o no intervino agente; no disting...,ID como cadena sin .0 y NA conservado; fuera d...
3,company,112593,No informado o no hay empresa asociada; no dis...,ID como cadena sin .0 y NA conservado; fuera d...


,tipo_original,tipo_limpio
children,float64,Int64
agent,float64,string
company,float64,string


,hotel,children,country,agent,company
40600,City Hotel,<NA>,PRT,<NA>,<NA>
40667,City Hotel,<NA>,PRT,14,<NA>
40679,City Hotel,<NA>,PRT,<NA>,<NA>
41160,City Hotel,<NA>,PRT,9,<NA>


**Decisión.** No se considera ausencia de agente/empresa equivalente a ID cero ni se afirma que signifique “no aplicable”. Se excluyen ambos de X hasta documentar semántica y versión inicial. Para país se reserva una categoría explícita de información desconocida, sin aprender la moda. Niños desconocidos no equivalen a cero niños: se mantiene el nulo y un indicador determinista, incluso si el entrenamiento futuro no contiene nulos. La mediana de niños será una aproximación estadística, no un valor recuperado.

Los conteos se tipan como enteros anulables en la copia limpia. Los predictores numéricos se pasan después a float64 para representar `np.nan` en scikit-learn; eso no cambia su significado de conteos. Se preservan `Undefined` y otras categorías originales, sin fusionarlas con ausencia sin documentación.

## 4. Anomalías: decisiones sin eliminación automática
**Pregunta:** ¿qué conocemos de estos casos y qué podemos corregir sin inventar?

In [4]:
# EDICIÓN COMENTADA PARA LECTURA. Las fuentes ejecutadas y selladas permanecen en la raíz.
# Cuantificar anomalías y conservarlas con sus metadatos de auditoría.
# Solo se añaden comentarios: no se han ejecutado instrucciones de esta copia.
nights = raw.stays_in_week_nights + raw.stays_in_weekend_nights
guests = raw[["adults", "children", "babies"]].sum(axis=1, min_count=3)
anomaly_policy = pd.DataFrame([
    ("ADR negativo", int(raw.adr.lt(0).sum()), "Podría ser ajuste o error; tarifa correcta desconocida", "Convertir a NaN solo en copia limpia; meta_adr_negative; ADR fuera de X."),
    ("ADR cero", int(raw.adr.eq(0).sum()), "Puede haber cortesía, acuerdos o ausencia de importe; no probado", "Conservar cero y reserva; meta_adr_zero; ADR fuera de X."),
    ("Cero noches", int(nights.eq(0).sum()), "Posible uso diurno, registro administrativo o error; no probado", "Conservar total_nights=0 y meta_zero_nights; sin imputar ni eliminar."),
    ("Cero huéspedes conocidos", int(guests.eq(0).sum()), "Posible reserva administrativa, agrupación o error; no probado", "Conservar composición y meta_zero_guests; no inventar huéspedes."),
    ("Huéspedes desconocidos", int(guests.isna().sum()), "children ausente impide calcular un total fiable", "No confundir con cero; meta_guests_unknown y children_missing.")
], columns=["problema", "registros", "posible_significado", "decision"])
display(anomaly_policy)
# Revisión de casos, no criterios de eliminación basados en el desenlace.
display(raw.loc[raw.adr.lt(0), ["hotel", "lead_time", "adr", "adults", "children", "stays_in_week_nights", "stays_in_weekend_nights"]])
# Hacer explícito este invariante: si falla, no se deben interpretar los resultados como válidos.
assert cleaned.loc[raw.adr.lt(0), "adr"].isna().all()
# Hacer explícito este invariante: si falla, no se deben interpretar los resultados como válidos.
assert cleaned.loc[raw.adr.eq(0), "adr"].eq(0).all()

,problema,registros,posible_significado,decision
0,ADR negativo,1,Podría ser ajuste o error; tarifa correcta des...,Convertir a NaN solo en copia limpia; meta_adr...
1,ADR cero,1959,"Puede haber cortesía, acuerdos o ausencia de i...",Conservar cero y reserva; meta_adr_zero; ADR f...
2,Cero noches,715,"Posible uso diurno, registro administrativo o ...",Conservar total_nights=0 y meta_zero_nights; s...
3,Cero huéspedes conocidos,180,"Posible reserva administrativa, agrupación o e...",Conservar composición y meta_zero_guests; no i...
4,Huéspedes desconocidos,4,children ausente impide calcular un total fiable,No confundir con cero; meta_guests_unknown y c...


,hotel,lead_time,adr,adults,children,stays_in_week_nights,stays_in_weekend_nights
14969,Resort Hotel,195,-6.38,2,0.0,6,4


**Decisión.** Se conservan las 119.390 filas. El ADR negativo se marca como no fiable sin adivinar una tarifa positiva; se conserva el original en `raw` y el indicador en el CSV preparado. Como ADR está excluido del conjunto conservador, su versión corregida no se exporta como predictor: no se genera otro CSV solo para guardarla. Los máximos de ADR y grandes grupos no se recortan por rareza ni mediante IQR. Las anomalías pueden solaparse: no se suman como si fueran reservas diferentes.

## 5. Auditoría temporal y selección conservadora
**Pregunta:** ¿qué información admitimos para predecir al crear la reserva?

In [5]:
# EDICIÓN COMENTADA PARA LECTURA. Las fuentes ejecutadas y selladas permanecen en la raíz.
# Justificar inclusiones y exclusiones por disponibilidad, no por puntuación predictiva.
# Solo se añaden comentarios: no se han ejecutado instrucciones de esta copia.
audit = audit_availability(raw.columns)
with pd.option_context("display.max_colwidth", None, "display.max_rows", 40):
    display(audit)
print("Exclusiones obligatorias:", LEAKAGE_FEATURES)
print("Exclusiones por incertidumbre:", UNCERTAIN_FEATURES)

,variable,disponibilidad,decision,justificacion
0,hotel,Disponible en la operación inicial (supuesto de trabajo),Incluir en X bajo supuesto explícito,Dato del plan de reserva/origen; el archivo no permite certificar su versión inicial.
1,is_canceled,No disponible (objetivo futuro),Etiqueta separada,No es una característica.
2,lead_time,Disponible en la operación inicial (supuesto de trabajo),Incluir en X bajo supuesto explícito,Dato del plan de reserva/origen; el archivo no permite certificar su versión inicial.
3,arrival_date_year,Disponible en la operación inicial (supuesto de trabajo),Excluir de X; calendario para auditoría o redundante,Dato del plan de reserva/origen; el archivo no permite certificar su versión inicial.
4,arrival_date_month,Disponible en la operación inicial (supuesto de trabajo),Incluir en X bajo supuesto explícito,Dato del plan de reserva/origen; el archivo no permite certificar su versión inicial.
5,arrival_date_week_number,Disponible en la operación inicial (supuesto de trabajo),Excluir de X; calendario para auditoría o redundante,Dato del plan de reserva/origen; el archivo no permite certificar su versión inicial.
6,arrival_date_day_of_month,Disponible en la operación inicial (supuesto de trabajo),Excluir de X; calendario para auditoría o redundante,Dato del plan de reserva/origen; el archivo no permite certificar su versión inicial.
7,stays_in_weekend_nights,Disponible en la operación inicial (supuesto de trabajo),Sustituir por total_nights en X,Dato del plan de reserva/origen; el archivo no permite certificar su versión inicial.
8,stays_in_week_nights,Disponible en la operación inicial (supuesto de trabajo),Sustituir por total_nights en X,Dato del plan de reserva/origen; el archivo no permite certificar su versión inicial.
9,adults,Disponible en la operación inicial (supuesto de trabajo),Incluir en X bajo supuesto explícito,Dato del plan de reserva/origen; el archivo no permite certificar su versión inicial.


Exclusiones obligatorias: ['reservation_status', 'reservation_status_date']
Exclusiones por incertidumbre: ['assigned_room_type', 'booking_changes', 'days_in_waiting_list', 'deposit_type', 'adr', 'total_of_special_requests', 'required_car_parking_spaces', 'previous_cancellations', 'previous_bookings_not_canceled', 'is_repeated_guest', 'agent', 'company']


**Decisión.** Estado final y fecha de estado nunca son predictores. Se excluyen además asignación, cambios, espera, depósito, ADR, solicitudes, aparcamiento, historial, repetición y los IDs de agente/empresa por no disponer de evidencia de su valor inicial.

Una política de depósito puede ser conocida al reservar: su exclusión es por incertidumbre de esta extracción, no por afirmar que siempre produce fuga. La espera depende de pre-reserva frente a confirmación; un contador final de cambios no existe al crear la reserva. Los contadores de historial solo serían válidos si se computaron con hechos conocidos antes de la operación actual.

Los campos seleccionados son datos que **el proceso de reserva necesita conocer inicialmente** (hotel, estancia prevista, composición, habitación reservada y origen comercial). Eso no certifica que esta extracción conserve su primera versión. Se asume explícitamente esa interpretación para preparar el conjunto reducido; si el diccionario revela actualizaciones, habrá que excluirlos o recuperar el snapshot inicial. Ningún subconjunto de este archivo permite certificar por sí solo ausencia total de fuga temporal.

## 6. Características y transformadores sin ajustar
**Pregunta:** ¿qué entra en X y qué se aprenderá solo con entrenamiento?

In [6]:
# EDICIÓN COMENTADA PARA LECTURA. Las fuentes ejecutadas y selladas permanecen en la raíz.
# Construir los 14 predictores y demostrar que el preprocesador sigue sin ajustar.
# Solo se añaden comentarios: no se han ejecutado instrucciones de esta copia.
features = build_features(cleaned)
display(pd.DataFrame({"variable": FEATURES, "tipo_modelado": ["numérica"] * len(NUMERIC_FEATURES) + ["categórica"] * len(CATEGORICAL_FEATURES)}))
display(features.head())
preprocessor = build_preprocessor(scale_numeric=True)
try:
    check_is_fitted(preprocessor)
except NotFittedError:
    print("Correcto: el transformador para reservas reales no está ajustado.")
else:
    raise AssertionError("No debe existir un transformador ajustado sobre datos reales.")
print(preprocessor)

,variable,tipo_modelado
0,lead_time,numérica
1,total_nights,numérica
2,adults,numérica
3,children,numérica
4,babies,numérica
5,children_missing,numérica
6,hotel,categórica
7,arrival_date_month,categórica
8,meal,categórica
9,country,categórica


,lead_time,total_nights,adults,children,babies,children_missing,hotel,arrival_date_month,meal,country,market_segment,distribution_channel,reserved_room_type,customer_type
0,342.0,0.0,2.0,0.0,0.0,0.0,Resort Hotel,July,BB,PRT,Direct,Direct,C,Transient
1,737.0,0.0,2.0,0.0,0.0,0.0,Resort Hotel,July,BB,PRT,Direct,Direct,C,Transient
2,7.0,1.0,1.0,0.0,0.0,0.0,Resort Hotel,July,BB,GBR,Direct,Direct,A,Transient
3,13.0,1.0,1.0,0.0,0.0,0.0,Resort Hotel,July,BB,GBR,Corporate,Corporate,A,Transient
4,14.0,2.0,2.0,0.0,0.0,0.0,Resort Hotel,July,BB,GBR,Online TA,TA/TO,A,Transient


Correcto: el transformador para reservas reales no está ajustado.
ColumnTransformer(sparse_threshold=1.0,
                  transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(keep_empty_features=True,
                                                                strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['lead_time', 'total_nights', 'adults',
                                  'children', 'babies', 'children_missing']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Desconocido',
                                                                keep_empty_features=True,
                                                                strategy='constant')),
      

**Transformaciones propuestas.**

- `total_nights`: suma de las dos clases de noches, sustituyéndolas para simplificar. No aprende parámetros.
- `children_missing`: indicador del valor desconocido antes de cualquier imputación. No aprende parámetros.
- Numéricas: `SimpleImputer(strategy='median')` y `StandardScaler` para algoritmos que lo necesiten. La fábrica admite `scale_numeric=False`. Si una columna fuera completamente ausente en entrenamiento, `keep_empty_features=True` la conserva con el relleno cero interno de scikit-learn: no sería una mediana aprendida; deberá revisarse ese caso en fase 4. El indicador evita confundir niños desconocidos con cero real.
- Categóricas: `SimpleImputer` con categoría constante `Desconocido`, seguido de `OneHotEncoder(handle_unknown='ignore')`. Una categoría nueva se representa con ceros en su bloque, sin error; no se aprende su efecto ni se equipara semánticamente con la ausencia.
- `ColumnTransformer(remainder='drop')` utiliza una lista explícita: etiqueta, metadatos y cualquier columna adicional quedan fuera. En las fases de modelos deberá integrarse en un `Pipeline` completo y ajustarse dentro de cada entrenamiento/fold, nunca antes del split.

No se incluyen año, semana y día de llegada en X: se reduce redundancia de calendario y dependencia de códigos/años específicos. Se conserva el mes como estacionalidad prevista; la fecha completa y el proxy se reservan a metadatos. No se crea `total_guests` como predictor para no duplicar componentes ni imputar niños al construirlo.

## 7. Estrategia temporal propuesta, sin crear particiones
**Pregunta:** ¿cómo ordenar reservas y evitar entrenar con resultados aún desconocidos?

In [7]:
# EDICIÓN COMENTADA PARA LECTURA. Las fuentes ejecutadas y selladas permanecen en la raíz.
# Mostrar la propuesta temporal original de fase 3, previa al protocolo definitivo v3.
# Solo se añaden comentarios: no se han ejecutado instrucciones de esta copia.
display(pd.Series(TEMPORAL_PLAN, name="regla").to_frame())

,regla
estado,propuesta pendiente de implementar y revisar e...
ordenacion,meta_booking_date_proxy; nunca equivale a crea...
corte_validacion,2016-07-01
corte_prueba,2017-01-01
entrenamiento,proxy < corte_validacion Y label_available_aft...
validacion,corte_validacion <= proxy < corte_prueba Y lab...
prueba,proxy >= corte_prueba; evaluar solo etiquetas ...
etiquetas,"label_available_after = max(llegada + noches, ..."
grupos,mantener fechas iguales juntas; verificar que ...
transformadores,fit únicamente en entrenamiento; transform en ...


**Propuesta reproducible para fase 4.** Ordenar por `llegada - lead_time`, reconociendo que es un **proxy** sin hora ni historial de cambios. Cortes propuestos: 01/07/2016 (inicio de validación) y 01/01/2017 (inicio de prueba). Se busca permitir un ciclo anual de llegadas antes del primer corte, dadas las llegadas desde julio de 2015. Son fronteras de calendario, no optimizadas mediante métricas; sus tamaños y cobertura por hotel se revisarán antes de adoptarlas.

Entrenamiento anterior al primer corte y validación entre cortes. Para cada simulación, no basta con que la reserva sea antigua: su etiqueta debe haber madurado antes del corte. Se propone exigir `max(llegada + noches, fecha del estado final) + 1 día < corte`. La fecha final interviene **solo como restricción retrospectiva de elegibilidad de la etiqueta**, nunca en X, imputación o codificación; no se usa para predecir. Esta cota conservadora no demuestra disponibilidad real en el sistema y puede excluir del entrenamiento reservas con cancelación ya conocida antes de su salida prevista.

En validación, usar solo resultados maduros antes del segundo corte para tomar decisiones que precedan a la prueba. En prueba, asegurar seguimiento suficiente. No dividir registros del mismo día entre conjuntos, verificar grupos duplicados y registrar cuántos casos quedan temporalmente sin asignar por falta de madurez. No hay muestreo aleatorio de duplicados. La fecha de corte y metadatos no llegan a X.

Una alternativa por llegadas solo mediría generalización a cohortes de llegada y no reproduciría una puesta en producción al reservar. La fidelidad histórica exige logs reales de creación, versiones de predictores y disponibilidad de resultados. El EDA ya ha visto todo el archivo: no se podrá presentar su prueba futura como totalmente ajena a exploración. No se implementa el split en esta fase.

## 8. Exportación de una única tabla preparada
**Pregunta:** ¿cómo conservar trazabilidad sin guardar múltiples datasets?

In [8]:
# EDICIÓN COMENTADA PARA LECTURA. Las fuentes ejecutadas y selladas permanecen en la raíz.
# Exportar preparación y verificar la separación entre X, y y metadatos.
# Solo se añaden comentarios: no se han ejecutado instrucciones de esta copia.
summary = run_preparation(PROJECT_ROOT)
prepared_path = PROJECT_ROOT / summary["archivo_preparado"]
# Leer las columnas y filas solicitadas; los filtros de esta llamada determinan la población disponible.
prepared = pd.read_csv(prepared_path, encoding="utf-8")
X, y, metadata = separate_model_data(prepared)
# Hacer explícito este invariante: si falla, no se deben interpretar los resultados como válidos.
assert list(X.columns) == FEATURES
# Hacer explícito este invariante: si falla, no se deben interpretar los resultados como válidos.
assert not (set(X.columns) & set(LEAKAGE_FEATURES + UNCERTAIN_FEATURES + ["is_canceled"]))
# Hacer explícito este invariante: si falla, no se deben interpretar los resultados como válidos.
assert not any(c.startswith("meta_") for c in X.columns)
# Comprobar tamaños esperados para detectar pérdidas de filas o desalineación.
assert len(X) == len(y) == len(metadata) == len(raw)
pd.testing.assert_series_equal(y, raw.is_canceled)
display(pd.Series({k: summary[k] for k in ["filas_originales", "filas_conservadas", "filas_excluidas", "duplicados_adicionales_conservados", "n_predictores", "columnas_exportadas"]}, name="resultado").to_frame())
display(X.isna().sum().to_frame("nulos_antes_de_ajustar"))
display(metadata.head())

,resultado
filas_originales,119390
filas_conservadas,119390
filas_excluidas,0
duplicados_adicionales_conservados,31994
n_predictores,14
columnas_exportadas,25


,nulos_antes_de_ajustar
lead_time,0
total_nights,0
adults,0
children,4
babies,0
children_missing,0
hotel,0
arrival_date_month,0
meal,0
country,488


,meta_row_id,meta_duplicate_group,meta_arrival_date,meta_booking_date_proxy,meta_label_available_after,meta_adr_negative,meta_adr_zero,meta_zero_nights,meta_zero_guests,meta_guests_unknown
0,0,692800ee3277792c,2015-07-01,2014-07-24,2015-07-02,0,1,1,0,0
1,1,2f9fbccb3d6527aa,2015-07-01,2013-06-24,2015-07-02,0,1,1,0,0
2,2,df72bae4c9559cbd,2015-07-01,2015-06-24,2015-07-03,0,0,0,0,0
3,3,d775b1f58f53265d,2015-07-01,2015-06-18,2015-07-03,0,0,0,0,0
4,4,4c474d8f4e4e036c,2015-07-01,2015-06-17,2015-07-04,0,0,0,0,0


**Resultado.** Un CSV preparado con **14 predictores + etiqueta + 10 metadatos = 25 columnas**, 119.390 filas y ninguna exclusión. Los 4 nulos de niños y 488 de país permanecen sin imputar. Los flags de anomalía no son predictores; sirven para revisar o evaluar sensibilidad más adelante. La huella de duplicados y `meta_label_available_after` dependen de información final: la lista cerrada de X evita que entren por accidente. No se debe construir X eliminando solo la etiqueta del CSV.

El CSV se regenera desde el original y no contiene matrices codificadas/escaladas ni transformadores persistidos. `separate_model_data` restaura tipos y valores `np.nan` después de leerlo; no hay una copia adicional por escenario.

## 9. Comprobaciones con datos sintéticos e integridad
**Pregunta:** ¿funcionan ausencia, categorías nuevas, aislamiento y reproducibilidad sin ajustar sobre el archivo completo?

In [9]:
# EDICIÓN COMENTADA PARA LECTURA. Las fuentes ejecutadas y selladas permanecen en la raíz.
# Comprobar las reglas de preparación y la conservación de los archivos de origen.
# Solo se añaden comentarios: no se han ejecutado instrucciones de esta copia.
# Seleccionar las pruebas de esta llamada; los conjuntos históricos pueden incluir entrenamientos.
suite = unittest.defaultTestLoader.discover(str(PROJECT_ROOT / "tests"), pattern="test_preparation.py")
log = io.StringIO()
# Ejecutar la batería indicada y conservar su salida; comprobar qué pruebas admite esta fase.
test_result = unittest.TextTestRunner(stream=log, verbosity=2).run(suite)
print(log.getvalue())
# Hacer explícito este invariante: si falla, no se deben interpretar los resultados como válidos.
assert test_result.wasSuccessful()
# El objeto real sigue sin fit después de las pruebas de objetos independientes.
try:
    check_is_fitted(preprocessor)
except NotFittedError:
    pass
else:
    raise AssertionError("El preprocesador real no debe estar ajustado.")
pd.testing.assert_frame_equal(raw, raw_before)
with DATA_PATH.open("rb") as file:
    # La huella debe conservarse: cualquier cambio de datos o evidencia detiene la comprobación.
    assert hashlib.file_digest(file, "sha256").hexdigest() == original_sha
references = json.loads((REPORT_PATH / "verificacion_referencias.json").read_text(encoding="utf-8"))
verified, unavailable = 0, 0
for file_name, expected in references["antes"].items():
    path = Path(file_name)
    if not path.is_file():
        unavailable += 1
        continue
    with path.open("rb") as file:
        # La huella debe conservarse: cualquier cambio de datos o evidencia detiene la comprobación.
        assert hashlib.file_digest(file, "sha256").hexdigest() == expected, file_name
    verified += 1
check_report = {"tests_sinteticos": test_result.testsRun, "fallos": len(test_result.failures),
    "errores": len(test_result.errors), "ajuste_solo_sintetico": True, "preprocesador_real_sin_fit": True,
    "raw_dataframe_intacto": True, "csv_original_intacto": True,
    "referencias_sin_cambios": verified, "referencias_no_disponibles": unavailable,
    "cwd_ejecucion": "raiz" if Path.cwd() == PROJECT_ROOT else "notebooks",
    "modelos_entrenados": False, "particiones_creadas": False,
    "sha256_preparado": summary["sha256_preparado"],
    "versiones": {"python": platform.python_version(), **{name: importlib.metadata.version(name) for name in ["pandas", "numpy", "scikit-learn", "nbformat", "nbclient", "ipykernel"]}}}
# Persistir documentación o metadatos para revisar las decisiones sin volver a ejecutar el modelo.
(REPORT_PATH / "preparacion_verificacion.json").write_text(json.dumps(check_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(pd.Series(check_report, name="verificacion").to_frame())

test_all_rows_and_duplicates_preserved_without_mutation (test_preparation.PreparationTests.test_all_rows_and_duplicates_preserved_without_mutation) ... ok
test_anomalies_and_unknowns_not_guessed (test_preparation.PreparationTests.test_anomalies_and_unknowns_not_guessed) ... ok
test_no_scaling_option_and_temporal_metadata (test_preparation.PreparationTests.test_no_scaling_option_and_temporal_metadata) ... ok
test_target_and_future_values_cannot_change_features (test_preparation.PreparationTests.test_target_and_future_values_cannot_change_features) ... ok
test_training_only_fit_unseen_categories_and_statistics (test_preparation.PreparationTests.test_training_only_fit_unseen_categories_and_statistics) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.134s

OK



,verificacion
tests_sinteticos,5
fallos,0
errores,0
ajuste_solo_sintetico,True
preprocesador_real_sin_fit,True
raw_dataframe_intacto,True
csv_original_intacto,True
referencias_sin_cambios,105
referencias_no_disponibles,0
cwd_ejecucion,raiz


**Conclusión de la fase.** Preparación determinista y separación explícita verificadas. Los tests usan solo datos inventados para comprobar medianas/medias aprendidas con entrenamiento sintético, no actualización al transformar datos posteriores, categorías inéditas, conservación de duplicados, anomalías, reproducibilidad y exclusión de etiqueta/metadatos/campos dudosos. No estiman utilidad predictiva ni certifican semántica temporal.

Antes de modelar se necesita autorizar la fase 4, revisar el contrato temporal con documentación del hotel/curso, validar cobertura y madurez en los cortes, asegurar ausencia de cruces por duplicados y crear las particiones. Después, cada ajuste estadístico se hará solo en el entrenamiento correspondiente. No hay modelos, optimización ni métricas predictivas en esta entrega.